<a href="https://colab.research.google.com/github/DanilRodenko/NLP-Job-Screener/blob/main/notebooks/03_Modeling_Matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df_pairs = pd.read_csv('/content/drive/MyDrive/NLP-Job-Screener/matching_pairs.csv')
df_resume = pd.read_csv('/content/drive/MyDrive/NLP-Job-Screener/resume_clean.csv')
df_jobs = pd.read_parquet('/content/drive/MyDrive/NLP-Job-Screener/jobs_clean.parquet')


In [ ]:
print(df_pairs.shape)
print(df_pairs.head(3))

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2', device="cuda")


In [ ]:
df_pairs.head(3)

In [ ]:
df_pairs['job_title_description'] = df_pairs['job_title'].str.lower() + " " + df_pairs['job_description']


resume_embeddings = model.encode(df_pairs['resume_text'].to_list(), convert_to_tensor=True)


job_embeddings = model.encode(df_pairs['job_title_description'].to_list(), convert_to_tensor=True)

In [ ]:
print(resume_embeddings.shape)
print(job_embeddings.shape)

In [ ]:
from sentence_transformers import util
cosine_scores = util.cos_sim(resume_embeddings, job_embeddings).diagonal()
print(cosine_scores.shape)
print(cosine_scores[:5])

In [ ]:
df_pairs['cosine_score'] = cosine_scores.cpu().numpy()

In [ ]:
df_pairs.head(10)

In [ ]:
def skill_overlap(resume_text, job_title_description):
  resume_text = set(resume_text.split())
  job_title_description = set(job_title_description.split())

  common = job_title_description.intersection(resume_text)
  ratio = len(common) / len(job_title_description) if job_title_description else 0
  return ratio

df_pairs['skill_overlap'] = df_pairs.apply(
    lambda row: skill_overlap(row['resume_text'], row['job_title_description']), axis=1
)

df_pairs.head(3)

In [ ]:
print(df_pairs['skill_overlap'].describe())

In [ ]:
def text_length_ratio(resume_text, job_title_description):
  if len(resume_text) > len(job_title_description):
    res = len(job_title_description) / len(resume_text)
  elif len(job_title_description) > len(resume_text):
    res = len(resume_text) / len(job_title_description)
  else:
    res = 1
  return res

In [ ]:
df_pairs['text_lenght_ratio'] = df_pairs.apply(
    lambda row: text_length_ratio(row['resume_text'], row['job_title_description']), axis=1
)
df_pairs.head()

In [ ]:
X = df_pairs[['cosine_score', 'skill_overlap', 'text_lenght_ratio']]
y = df_pairs['label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.3
)

In [ ]:
from sklearn.utils import class_weight
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=3,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
import numpy as np

resume_np = resume_embeddings.cpu().numpy()
job_np = job_embeddings.cpu().numpy()

diff = np.abs(resume_np - job_np)

X_emb = np.hstack([
    df_pairs[['cosine_score', 'skill_overlap', 'text_lenght_ratio']].values,
    diff
])

X_train, X_test, y_train, y_test = train_test_split(
    X_emb,
    y,
    random_state=42,
    test_size=0.3
)

from sklearn.utils import class_weight
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=3,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
import joblib
joblib.dump(model, 'matching_model.joblib')